# OpenAI Agents SDK + Email Infrastructure

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/openai_agents_email.ipynb)

Add email capabilities to an OpenAI Agents SDK agent using `@function_tool`. The agent gets `send_email`, `read_inbox`, `search_threads`, and `reply_to_thread` as tools it can call during its reasoning loop.

**What you'll build:** A support agent that:
1. Reads inbound emails from its inbox
2. Searches thread history for context on the customer
3. Generates a personalized reply based on history
4. Sends the reply in the existing thread

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier)
- [OpenAI API key](https://platform.openai.com)

In [ ]:
!pip install commune-mail openai-agents -q
print("\u2713 Dependencies installed")

## Setup

Initialize the Commune client and create a dedicated inbox for the agent. We also configure the OpenAI client — the Agents SDK picks up `OPENAI_API_KEY` from the environment automatically.

In [ ]:
import os
import json
from commune import CommuneClient

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-your_key_here")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # agents SDK reads from env

commune = CommuneClient(api_key=COMMUNE_API_KEY)

# Create the agent's inbox
agent_inbox = commune.inboxes.create(local_part="openai-support-agent")

print(f"\u2713 Agent inbox ready: {agent_inbox.address}")
print(f"  Inbox ID: {agent_inbox.id}")

## Define Email Tools with `@function_tool`

The OpenAI Agents SDK uses `@function_tool` to expose Python functions as tools the agent can call. The function's docstring becomes the tool description — write it clearly so the model understands when and how to use each tool.

Type hints are required: the SDK generates the JSON schema for the tool from the function signature.

In [ ]:
from agents import function_tool

INBOX_ID = agent_inbox.id  # captured once at setup


@function_tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a new email from the agent's inbox to a recipient.

    Use this to start a new email conversation. For replies to existing
    conversations, use reply_to_thread instead.

    Args:
        to: Recipient email address (e.g. 'customer@example.com').
        subject: Subject line for the email.
        body: Plain-text body of the email.

    Returns:
        A confirmation string with the message ID.
    """
    result = commune.messages.send(
        to=to,
        subject=subject,
        text=body,
        inbox_id=INBOX_ID,
    )
    return f"Email sent successfully. Message ID: {result.get('message_id', 'ok')}"


@function_tool
def reply_to_thread(thread_id: str, to: str, body: str) -> str:
    """Reply to an existing email thread.

    Always use this (not send_email) when responding to a customer's inbound
    email. This ensures your reply appears in the same conversation thread in
    the customer's email client, preserving context.

    Args:
        thread_id: The Commune thread ID (from read_inbox or webhook payload).
        to: The customer's email address to reply to.
        body: Plain-text reply body.

    Returns:
        A confirmation string with the message ID.
    """
    result = commune.messages.send(
        to=to,
        text=body,
        inbox_id=INBOX_ID,
        thread_id=thread_id,
    )
    return f"Reply sent in thread {thread_id}. Message ID: {result.get('message_id', 'ok')}"


@function_tool
def read_inbox(limit: int = 10) -> str:
    """List the most recent email threads in the agent's inbox.

    Call this to see what emails are waiting for a response. Returns a formatted
    list of threads with their IDs, subjects, senders, and message counts.

    Args:
        limit: How many threads to return. Maximum 50.

    Returns:
        A formatted list of threads, or a message if the inbox is empty.
    """
    threads = commune.threads.list(inbox_id=INBOX_ID, limit=limit)
    if not threads.data:
        return "Inbox is empty — no threads found."

    lines = [f"Found {len(threads.data)} thread(s):\n"]
    for t in threads.data:
        lines.append(
            f"- Thread ID: {t.thread_id}\n"
            f"  Subject: {t.subject or '(no subject)'}\n"
            f"  Messages: {t.message_count}\n"
            f"  Last active: {t.last_message_at}\n"
        )
    return "\n".join(lines)


@function_tool
def search_threads(query: str, limit: int = 5) -> str:
    """Search past email threads using natural language.

    Use this before replying to a customer to find any prior conversations
    they may have had. Results are ranked by semantic similarity.

    Args:
        query: A natural language description of what you're looking for,
               e.g. 'billing dispute from Alice' or 'password reset issue'.
        limit: Maximum number of results to return.

    Returns:
        A list of matching threads sorted by relevance.
    """
    results = commune.search.threads(
        query=query,
        inbox_id=INBOX_ID,
        limit=limit,
    )
    if not results:
        return "No matching threads found."

    lines = [f"Found {len(results)} matching thread(s):\n"]
    for r in results:
        lines.append(
            f"- Thread ID: {r.thread_id}\n"
            f"  Subject: {r.subject}\n"
            f"  Relevance score: {r.score:.2f}\n"
        )
    return "\n".join(lines)


@function_tool
def get_thread_messages(thread_id: str) -> str:
    """Get all messages in a specific email thread.

    Use this to read the full conversation history before drafting a reply.
    Returns messages in chronological order with sender and content.

    Args:
        thread_id: The Commune thread ID to fetch messages from.

    Returns:
        All messages in the thread, formatted for readability.
    """
    messages = commune.threads.messages(thread_id)
    if not messages:
        return f"No messages found in thread {thread_id}."

    lines = [f"Thread {thread_id} — {len(messages)} message(s):\n"]
    for i, msg in enumerate(messages, 1):
        sender = next(
            (p.identity for p in msg.participants if p.role == "sender"),
            "unknown"
        )
        direction = "\u2192 Sent" if msg.direction == "outbound" else "\u2190 Received"
        lines.append(
            f"{i}. {direction} | From: {sender}\n"
            f"   {msg.content[:300]}{'...' if len(msg.content) > 300 else ''}\n"
        )
    return "\n".join(lines)


email_tools = [send_email, reply_to_thread, read_inbox, search_threads, get_thread_messages]
print(f"\u2713 {len(email_tools)} function tools defined:")
for t in email_tools:
    print(f"  - {t.name}")

## Create the OpenAI Agent

We use the OpenAI Agents SDK `Agent` class with `gpt-4o-mini`. The `instructions` field is the system prompt — it tells the agent its role, constraints, and which tools to call in which order.

The key instruction is: **always call `search_threads` before drafting a reply** — this gives the agent customer history without extra code.

In [ ]:
from agents import Agent

support_agent = Agent(
    name="EmailSupportAgent",
    model="gpt-4o-mini",
    instructions="""You are a customer support agent with access to an email inbox.

Your workflow for every support request:
1. Call search_threads to find any prior conversations with this customer
2. If relevant prior threads exist, call get_thread_messages to read the history
3. Draft a reply that:
   - Acknowledges the specific issue (not a template)
   - References relevant history if you found any
   - Provides a concrete solution or clear next step
   - Ends with an open offer to help further
4. Always use reply_to_thread (not send_email) for replies — this keeps the thread intact

Tone: Warm, professional, and specific. Never generic. Under 150 words per reply.

If you receive a task that says 'simulation', draft the reply but do not send it.
""",
    tools=email_tools,
)

print(f"\u2713 Agent '{support_agent.name}' ready")
print(f"  Model: {support_agent.model}")
print(f"  Tools: {[t.name for t in support_agent.tools]}")

## Run the Agent with a Sample Email

We use `Runner.run()` to invoke the agent with an inbound email payload. The `Runner` manages the tool call loop — it runs the agent, executes any tool calls the model makes, and feeds results back until the agent produces a final answer.

Set `simulation=True` in the task to prevent the agent from actually sending the email while testing.

In [ ]:
import asyncio
from agents import Runner

# Simulated inbound email (same structure as a Commune webhook payload)
inbound_email = {
    "sender": "jane@example.com",
    "subject": "My export is stuck at 0% for 2 hours",
    "thread_id": "thread_demo_openai_001",
    "content": (
        "Hi, I started a data export two hours ago and it's still showing 0%. "
        "I have a deadline in 3 hours and I need this data for a client presentation. "
        "Can you please look into this urgently? My account is jane@example.com."
    ),
}

task = (
    f"New inbound support email (SIMULATION — do not actually send a reply):\n\n"
    f"From: {inbound_email['sender']}\n"
    f"Subject: {inbound_email['subject']}\n"
    f"Thread ID: {inbound_email['thread_id']}\n\n"
    f"Content:\n{inbound_email['content']}\n\n"
    f"Search for any prior threads from this customer, then draft the reply "
    f"you would send. Show me the final reply text."
)

async def run_support_agent(task_text: str):
    result = await Runner.run(support_agent, task_text)
    return result

print("Running agent...")
print("=" * 60)

# In a notebook, use asyncio.run() or await depending on your environment
try:
    # Jupyter environments have a running event loop
    result = await Runner.run(support_agent, task)
except RuntimeError:
    # Fallback for standard Python environments
    result = asyncio.run(run_support_agent(task))

print("=" * 60)
print("\nFinal answer:")
print(result.final_output)

## Production Webhook Handler

In production, the agent runs automatically whenever an email arrives. Here is a complete FastAPI webhook handler that verifies the Commune signature and invokes the agent asynchronously.

In [ ]:
webhook_code = '''
import os
import json
import asyncio
from fastapi import FastAPI, Request, Response
from commune import verify_signature, WebhookVerificationError
from agents import Runner

app = FastAPI()

@app.post("/webhook")
async def handle_inbound(request: Request):
    body = await request.body()

    # Verify the Commune webhook signature
    try:
        verify_signature(
            payload=body,
            signature=request.headers.get("x-commune-signature", ""),
            secret=os.environ["COMMUNE_WEBHOOK_SECRET"],
            timestamp=request.headers.get("x-commune-timestamp", ""),
        )
    except WebhookVerificationError as e:
        return Response(content=str(e), status_code=401)

    payload = json.loads(body)
    sender = payload["sender"]
    subject = payload.get("subject", "(no subject)")
    thread_id = payload["thread_id"]
    content = payload["content"]

    # Build the agent task from the inbound email
    task = (
        f"New inbound support email:\n\n"
        f"From: {sender}\n"
        f"Subject: {subject}\n"
        f"Thread ID: {thread_id}\n\n"
        f"Content:\n{content}\n\n"
        f"Search for prior context from this customer, then reply_to_thread."
    )

    # Run agent in background so we can return 200 immediately
    asyncio.create_task(Runner.run(support_agent, task))

    return {"ok": True}
'''

print("Production FastAPI webhook handler:")
print(webhook_code)

## Inspect Tool Calls

The `RunResult` object contains the full trace of tool calls made during the agent's run. Useful for debugging and for building observability into your agent pipeline.

In [ ]:
# Inspect what tools the agent called (from the previous run)
if result and hasattr(result, 'new_items'):
    print("Tool calls made during agent run:")
    for item in result.new_items:
        item_type = type(item).__name__
        if "ToolCall" in item_type or "tool_call" in str(item_type).lower():
            print(f"  Tool: {getattr(item, 'name', item_type)}")
            if hasattr(item, 'arguments'):
                args = item.arguments
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except Exception:
                        pass
                print(f"  Args: {args}")
            print()
else:
    print("Run result items not available — the agent may have completed without tool calls.")
    print(f"Final output:\n{result.final_output if result else 'N/A'}")

## What's next?

- **Add handoffs** — use the Agents SDK `handoff()` to route to a human agent when the issue is too complex
- **Add guardrails** — use `input_guardrail` to filter out spam before the agent processes the email
- **Multi-agent** — combine with CrewAI or a second OpenAI agent for triage → specialist workflows
- **[Structured extraction](./structured_extraction.ipynb)** — attach a schema to the inbox so the agent receives pre-parsed fields
- **[LangChain notebook](./langchain_customer_support.ipynb)** — LangChain alternative with `@tool` decorator

**OpenAI Agents SDK docs:** https://openai.github.io/openai-agents-python  
**Commune docs:** https://commune.email/docs